Code Overview:
This script provides a user-friendly approach for researchers without GPU resources to perform urban mapping globally.
Before executing this script, ensure you have met the following prerequisites:
- Acquired multimodal imagery for your target study area according to the instructions provided at: https://github.com/LauraChow77/GlobalUrbanMapper2.0/tree/main/gee_code
- Obtained the Global Urban Mapper 2.0 (GUM 2.0) model checkpoint.
- Configure your Google Colab notebook to use a T4 GPU with the appropriate Python version by going to 'Runtime' -> 'Change runtime type'. In the pop-up window, please ensure that the 'Runtime type' is set to 'Python 3', the 'Hardware accelerator' is set to 'T4 GPU', and the 'Runtime version' is set to '2025.07'.

Assistance:
If you encounter any issues or have questions, please do not hesitate to contact me at 22042458r@connect.polyu.hk

# Set Up

Setup Section:
This section prepares the environment for the global urban mapping task by:
- Installing necessary dependencies.
- Importing required libraries.
- Authenticating Google Drive access for file retrieval (e.g., multimodal imagery, model files) and for saving model predictions.

## Install dependencies

Install the specified PyTorch version first to avert getting stuck while building mmcv wheels. (https://github.com/open-mmlab/mmdetection/issues/6909)

In [ ]:
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

Please restart the runtime after executing the cell BELOW. Navigate to 'Runtime' in the menu and select 'Restart session'.

In [ ]:
!pip install numpy==1.26.4
!pip install rasterio
!pip install ftfy
!pip3 install openmim
!mim install mmengine
!mim install "mmcv>=2.0.0"
!pip install mmsegmentation

Please restart the runtime after executing the cell ABOVE. Navigate to 'Runtime' in the menu and select 'Restart session'.

## Import libraries

ATTENTION: mmcv just released a new version but the downstream libraries (e.g., mmsegmentation) are not yet supported. You might encounter issues such as:

AssertionError: MMCV==2.2.0 is used but incompatible. Please install mmcv>=2.0.0rc4.

To handle this issue, open /usr/local/lib/python3.10/dist-packages/mmseg/__init__.py file and comment out the related assert code (l61-l63). This could solve the issue! (https://github.com/open-mmlab/mmcv/issues/3096)

In [ ]:
import ee
from google.colab import auth

import os

import warnings
from affine import Affine
import rasterio
import numpy as np
from tqdm import tqdm

import timm

import torch
from torchvision import transforms
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torchvision


from mmengine.registry import init_default_scope
from mmengine import Config
from mmseg.apis import inference_model, init_model
from mmseg.models import build_segmentor

from mmengine.registry import init_default_scope
from mmengine.model import BaseModule
from mmcv.cnn import ConvModule, build_upsample_layer
from mmseg.structures import build_pixel_sampler
from mmseg.models.utils import resize

from abc import ABCMeta, abstractmethod

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Global Variables

In [ ]:
""" Model inference variables"""
BATCH_SIZE = 1
NUM_WORKERS = 16
with_product = False
INPUT_CHANNELS = 13 if with_product else 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'GUM2.0'

CKPT_PATH = '/content/drive/My Drive/GUM2.0/model/GUM_2p0.pth'      # the model path
INFERENCE_DATASET_DIR = '/content/drive/My Drive/GUM2.0/datasets'     # the multimodal image(s) directory
PREDICTED_DATASET_DIR = '/content/drive/My Drive/GUM2.0/output'     # the predicted image directory

data_transforms = transforms.Compose([
    transforms.ToTensor(),
])

# Dataset

In [ ]:
class GUM(Dataset):
    def __init__(self, root_dir, with_product=False, transform=None, inference_mode=False):
        """
        Args:
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
            inference_mode (bool, optional): Flag to indicate whether the dataset is used for inference.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.with_product = with_product
        self.inference_mode = inference_mode

        self.data = []

        if not self.inference_mode:  # Training/Validation mode
            for cls_name in self.class_names:
                cls_folder = os.path.join(root_dir, cls_name)
                for img_filename in os.listdir(cls_folder):
                    if img_filename.endswith('.tif'):
                        img_path = os.path.join(cls_folder, img_filename)
                        label_path = img_path.replace('img_stack_with_product', 'label')
                        self.data.append((img_path, label_path))
        else:  # Inference mode
            if os.path.isdir(root_dir):
                for img_filename in os.listdir(root_dir):
                    if img_filename.endswith('.tif'):
                        img_path = os.path.join(root_dir, img_filename)
                        self.data.append((img_path, None))
            else:
                raise FileNotFoundError(f"The directory {root_dir} does not exist.")

    def __len__(self):
        return len(self.data)

    @staticmethod
    def _normalize_per_channel(data, vmin, vmax):
        """Replace NaNs/inf per channel, clip, and min-max normalize to [0, 1]."""
        data = data.astype(np.float32)
        for i in range(data.shape[0]):
            data[i] = np.nan_to_num(
                data[i], nan=vmax[i], posinf=vmax[i], neginf=vmin[i])
        vmin = vmin.reshape(-1, 1, 1)
        vmax = vmax.reshape(-1, 1, 1)
        data = np.clip(data, vmin, vmax)
        return (data - vmin) / (vmax - vmin)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]

        with rasterio.open(img_path) as data:
            multimodal_data = data.read()

        multimodal_data = np.nan_to_num(multimodal_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Handle s2 data
        multimodal_data[0:4] = _normalize_scalar(multimodal_data[0:4], vmin=0.0, vmax=1.0, nan=0.0)

        # Handle s1 data
        # S1: 4 SAR bands, clipped to per-band 0.5-99.5% thresholds.
        s1_clip_min = np.array([-23.62251091003418, -37.2230110168457,
                                -24.20653533935547, -37.36018753051758], dtype=np.float32)
        s1_clip_max = np.array([2.138315200805664, 0.0,
                                2.071335747242074, 0.0], dtype=np.float32)
        multimodal_data[4:8] = self._normalize_per_channel(multimodal_data[4:8], s1_clip_min, s1_clip_max)

        # Handle dem data as per the provided code snippet
        slope = _normalize_scalar(multimodal_data[9:10], vmin=0.0, vmax=90.0, nan=0.0)
        aspect = _normalize_scalar(multimodal_data[10:11], vmin=0.0, vmax=360.0, nan=0.0)
        multimodal_data[8:10] = np.concatenate((slope, aspect), axis=0)

        if self.with_product:
            multimodal_data[10:13] = multimodal_data[11:14]
            img = np.transpose(multimodal_data[:13, :, :], (1, 2, 0))
        else:
            img = np.transpose(multimodal_data[:10, :, :], (1, 2, 0))

        # Apply the transformations
        if self.transform:
            img = self.transform(img)  # img is a numpy array and will be converted to a tensor here

        file_name = os.path.basename(img_path)
        results = {}
        results['filename'] = file_name
        results['ori_filename'] = file_name
        results['img'] = img
        results['img_shape'] = img.shape
        results['ori_shape'] = img.shape
        # Set initial values for default meta_keys
        results['pad_shape'] = img.shape
        results['scale_factor'] = 1.0
        num_channels = 1 if len(img.shape) < 3 else img.shape[2]
        results['img_norm_cfg'] = dict(
            mean=np.zeros(num_channels, dtype=np.float32),
            std=np.ones(num_channels, dtype=np.float32),
            to_rgb=False)
        results['flip'] = False
        # For inference, return the image and the file name
        if self.inference_mode:
            return img, results  # Return the image path instead of the label
        else:
            return img, label

def _normalize_scalar(data, vmin, vmax, nan):
        """Replace NaNs/inf, clip, and min-max normalize to [0, 1]."""
        data = np.nan_to_num(data, nan=nan, posinf=nan, neginf=vmin)
        return np.clip((data - vmin) / (vmax - vmin), 0.0, 1.0)

# Models

## MultiEncUNet

In [ ]:
class UNetResMultiEnc(BaseModule):
    def __init__(self,
                 s2_in_channels=4, s1_in_channels=4, topo_in_channels=2,
                 base_channels=64,
                 num_stages=5,
                 strides=(1, 1, 1, 1, 1),
                 enc_num_convs=(2, 2, 2, 2, 2),
                 dec_num_convs=(2, 2, 2, 2),
                 downsamples=(True, True, True, True),
                 enc_dilations=(1, 1, 1, 1, 1),
                 dec_dilations=(1, 1, 1, 1),
                 with_cp=False,
                 conv_cfg=None,
                 norm_cfg=dict(type='BN'),
                 act_cfg=dict(type='ReLU'),
                 upsample_cfg=dict(type='InterpConv'),
                 norm_eval=False,
                 dcn=None,
                 plugins=None,
                 pretrained=None,
                 init_cfg=None,
                 backbone_name='swsl_resnet18',
                 num_classes=2):
        super(UNetResMultiEnc, self).__init__(init_cfg)

        self.pretrained = pretrained
        assert not (init_cfg and pretrained), \
            'init_cfg and pretrained cannot be setting at the same time'
        if isinstance(pretrained, str):
            warnings.warn('DeprecationWarning: pretrained is a deprecated, '
                          'please use "init_cfg" instead')
            self.init_cfg = dict(type='Pretrained', checkpoint=pretrained)
        elif pretrained is None:
            if init_cfg is None:
                self.init_cfg = [
                    dict(type='Kaiming', layer='Conv2d'),
                    dict(
                        type='Constant',
                        val=1,
                        layer=['_BatchNorm', 'GroupNorm'])
                ]
        else:
            raise TypeError('pretrained must be a str or None')

        assert dcn is None, 'Not implemented yet.'
        assert plugins is None, 'Not implemented yet.'
        assert len(strides) == num_stages, \
            'The length of strides should be equal to num_stages, '\
            f'while the strides is {strides}, the length of '\
            f'strides is {len(strides)}, and the num_stages is '\
            f'{num_stages}.'
        assert len(enc_num_convs) == num_stages, \
            'The length of enc_num_convs should be equal to num_stages, '\
            f'while the enc_num_convs is {enc_num_convs}, the length of '\
            f'enc_num_convs is {len(enc_num_convs)}, and the num_stages is '\
            f'{num_stages}.'
        assert len(dec_num_convs) == (num_stages-1), \
            'The length of dec_num_convs should be equal to (num_stages-1), '\
            f'while the dec_num_convs is {dec_num_convs}, the length of '\
            f'dec_num_convs is {len(dec_num_convs)}, and the num_stages is '\
            f'{num_stages}.'
        assert len(downsamples) == (num_stages-1), \
            'The length of downsamples should be equal to (num_stages-1), '\
            f'while the downsamples is {downsamples}, the length of '\
            f'downsamples is {len(downsamples)}, and the num_stages is '\
            f'{num_stages}.'
        assert len(enc_dilations) == num_stages, \
            'The length of enc_dilations should be equal to num_stages, '\
            f'while the enc_dilations is {enc_dilations}, the length of '\
            f'enc_dilations is {len(enc_dilations)}, and the num_stages is '\
            f'{num_stages}.'
        assert len(dec_dilations) == (num_stages-1), \
            'The length of dec_dilations should be equal to (num_stages-1), '\
            f'while the dec_dilations is {dec_dilations}, the length of '\
            f'dec_dilations is {len(dec_dilations)}, and the num_stages is '\
            f'{num_stages}.'
        self.num_stages = num_stages
        self.strides = strides
        self.downsamples = downsamples
        self.norm_eval = norm_eval
        self.base_channels = base_channels
        self.num_classes = num_classes

        self.s2_encoder = ResEncoder(in_channels=s2_in_channels, backbone_name=backbone_name, pretrained=False)
        self.s1_encoder = ResEncoder(in_channels=s1_in_channels, backbone_name=backbone_name, pretrained=False)
        self.topo_encoder = ResEncoder(in_channels=topo_in_channels, backbone_name=backbone_name, pretrained=False)

        encoder_channels = self.s2_encoder.get_channels()

        self.ffm = nn.ModuleList()

        self.decoder = nn.ModuleList()
        self.decode_head = PSPHead(in_channels=64,
                                in_index=4,
                                channels=16,
                                pool_scales=(1, 2, 3, 6),
                                dropout_ratio=0.1,
                                num_classes=num_classes,
                                norm_cfg=norm_cfg,
                                align_corners=False)

        self.decoder = nn.ModuleList()
        for i in range(num_stages):
            if i != 0:
                upsample = (strides[i] != 1 or downsamples[i - 1])
                self.decoder.append(
                    UpConvBlock(
                        conv_block=BasicConvBlock,
                        in_channels=encoder_channels[i],
                        skip_channels=encoder_channels[i-1],
                        out_channels=encoder_channels[i-1],
                        num_convs=dec_num_convs[i - 1],
                        stride=1,
                        dilation=dec_dilations[i - 1],
                        with_cp=with_cp,
                        conv_cfg=conv_cfg,
                        norm_cfg=norm_cfg,
                        act_cfg=act_cfg,
                        upsample_cfg=upsample_cfg if upsample else None,
                        dcn=None,
                        plugins=None))
            self.ffm.append(MultimodalConcat(encoder_channels[i], scale_aligned=True))

    def forward(self, s2, s1, topo):
        self._check_input_divisible(s2)
        self._check_input_divisible(s1)
        self._check_input_divisible(topo)

        s2_enc_outs = self.s2_encoder(s2)
        s1_enc_outs = self.s1_encoder(s1)
        topo_enc_outs = self.topo_encoder(topo)
        s2, s1, topo = s2_enc_outs[-1], s1_enc_outs[-1], topo_enc_outs[-1]
        x = self.ffm[self.num_stages-1](s2, s1, topo)
        dec_outs = [x]
        fuse_cache = [x]
        for i in reversed(range(len(self.decoder))):
            enc_outs = self.ffm[i](s2_enc_outs[i], s1_enc_outs[i], topo_enc_outs[i])
            fuse_cache.append(enc_outs)
            x = self.decoder[i](enc_outs, x)
            dec_outs.append(x)

        output = self.decode_head(dec_outs)

        return output

    def _check_input_divisible(self, x):
        h, w = x.shape[-2:]
        whole_downsample_rate = 1
        for i in range(1, self.num_stages):
            if self.strides[i] == 2 or self.downsamples[i - 1]:
                whole_downsample_rate *= 2
        assert (h % whole_downsample_rate == 0) \
            and (w % whole_downsample_rate == 0),\
            f'The input image size {(h, w)} should be divisible by the whole '\
            f'downsample rate {whole_downsample_rate}, when num_stages is '\
            f'{self.num_stages}, strides is {self.strides}, and downsamples '\
            f'is {self.downsamples}.'

    def test_step(self, s2, s1, topo, patch_size=(256, 256), overlap=0):
        """
        Divide a large input matrix into smaller patches with overlap, perform inference on each patch,
        and combine the results to produce a single-channel output of shape (H, W).

        Args:
            s2 (torch.Tensor): Input tensor for S2 data of shape (1, C, H, W).
            s1 (torch.Tensor): Input tensor for S1 data of shape (1, C, H, W).
            topo (torch.Tensor): Input tensor for topographic data of shape (1, C, H, W).
            patch_size (tuple): Size of each patch (height, width). Must be divisible by 16.
            overlap (int): Overlap between patches to avoid edge artifacts.

        Returns:
            torch.Tensor: Combined output tensor of shape (H, W) representing predicted class indices.
        """
        # Ensure patch size is divisible by 16
        assert patch_size[0] % 16 == 0 and patch_size[1] % 16 == 0, \
            "Patch size must be divisible by 16 to meet the model's downsample rate requirement."

        # Get input dimensions
        _, C, H, W = s2.shape

        # Initialize an empty tensor to store the final output
        output = torch.zeros((1, self.num_classes, H, W), device=s2.device)

        # Initialize a weight matrix to handle overlapping regions
        weight_matrix = torch.zeros((1, 1, H, W), device=s2.device)

        # Define patch size and stride (accounting for overlap)
        patch_h, patch_w = patch_size
        stride_h, stride_w = patch_h - overlap, patch_w - overlap

        # Iterate over the input tensor in patches with overlap
        for i in range(0, H, stride_h):
            for j in range(0, W, stride_w):
                # Calculate patch boundaries
                h_start, h_end = i, min(i + patch_h, H)
                w_start, w_end = j, min(j + patch_w, W)

                # If the patch is smaller than the patch size, extend it by including previous pixels
                if (h_end - h_start) < patch_h:
                    h_start = max(0, h_end - patch_h)
                if (w_end - w_start) < patch_w:
                    w_start = max(0, w_end - patch_w)

                # Extract the patch
                s2_patch = s2[:, :, h_start:h_end, w_start:w_end]
                s1_patch = s1[:, :, h_start:h_end, w_start:w_end]
                topo_patch = topo[:, :, h_start:h_end, w_start:w_end]

                # Perform inference on the patch
                with torch.no_grad():
                    patch_output = self.forward(s2_patch, s1_patch, topo_patch)  # Replace with your model's forward pass

                # Place the patch output in the corresponding location in the final output
                output[:, :, h_start:h_end, w_start:w_end] += patch_output
                weight_matrix[:, :, h_start:h_end, w_start:w_end] += 1

        # Normalize the output by dividing by the weight matrix to handle overlapping regions
        output /= weight_matrix

        # Convert multi-channel output to single-channel by taking the argmax along the class dimension
        single_channel_output = torch.argmax(output, dim=1)  # Shape: (1, H, W)

        # Remove the batch dimension to return a tensor of shape (H, W)
        return single_channel_output.squeeze(0)

class BasicConvBlock(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels,
                 num_convs=2,
                 stride=1,
                 dilation=1,
                 with_cp=False,
                 conv_cfg=None,
                 norm_cfg=dict(type='BN'),
                 act_cfg=dict(type='ReLU'),
                 dcn=None,
                 plugins=None,
                 dws=False):
        super(BasicConvBlock, self).__init__()
        assert dcn is None, 'Not implemented yet.'
        assert plugins is None, 'Not implemented yet.'

        self.with_cp = with_cp
        convs = []
        for i in range(num_convs):
            convs.append(
                ConvModule(
                    in_channels=in_channels if i == 0 else out_channels,
                    out_channels=out_channels,
                    kernel_size=3,
                    stride=stride if i == 0 else 1,
                    dilation=1 if i == 0 else dilation,
                    padding=1 if i == 0 else dilation,
                    conv_cfg=conv_cfg,
                    norm_cfg=norm_cfg,
                    act_cfg=act_cfg,
                    groups=in_channels if dws else 1))

        self.convs = nn.Sequential(*convs)

    def forward(self, x):
        if self.with_cp and x.requires_grad:
            out = cp.checkpoint(self.convs, x)
        else:
            out = self.convs(x)
        return out


class ResEncoder(nn.Module):
    def __init__(self,
                 in_channels=10,
                 decode_channels=64,
                 dropout=0.1,
                 backbone_name='swsl_resnet18',
                 pretrained=True,
                 window_size=8,
                 num_classes=2
                 ):
        super().__init__()

        self.initial_block = BasicConvBlock(
                    in_channels=in_channels,
                    out_channels=64,
                    num_convs=2,
                    stride=1,
                    dilation=1,
                    with_cp=False,
                    conv_cfg=None,
                    norm_cfg=dict(type='BN'),
                    act_cfg=dict(type='ReLU'),
                    dcn=None,
                    plugins=None)

        self.encoder = timm.create_model(backbone_name, features_only=True, output_stride=16,
                                          out_indices=(0, 1, 2, 3), pretrained=True)
        original_first_conv = self.encoder.conv1
        self.encoder.conv1 = nn.Conv2d(
            in_channels=64,
            out_channels=original_first_conv.out_channels,
            kernel_size=original_first_conv.kernel_size,
            stride=original_first_conv.stride,
            padding=original_first_conv.padding,
            bias=False
        )

        if pretrained:
            pretrained_state_dict = timm.create_model(backbone_name, pretrained=True).state_dict()
            if 'conv1.weight' in pretrained_state_dict:
                del pretrained_state_dict['conv1.weight']
            if 'conv1.bias' in pretrained_state_dict:
                del pretrained_state_dict['conv1.bias']
            self.encoder.load_state_dict(pretrained_state_dict, strict=False)

    def forward(self, x):
        x = self.initial_block(x)
        features = self.encoder(x)
        return [x] + features

    def get_channels(self):
        return [64] + self.encoder.feature_info.channels()


class MultimodalConcat(nn.Module):
    def __init__(self, n_channels, is_full=True, scale_aligned=False):
        super(MultimodalConcat, self).__init__()
        self.scale_aligned = scale_aligned
        if scale_aligned:
            self.s2_scale = nn.BatchNorm2d(n_channels)
            self.s1_scale = nn.BatchNorm2d(n_channels)
            self.srtm_scale = nn.BatchNorm2d(n_channels)
        ratio = 2
        if is_full:
            ratio += 1
        self.out = nn.Sequential(
            nn.Conv2d(n_channels * ratio, n_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(n_channels),
            nn.ReLU()
        )

    def forward(self, s2, s1, topo=None):
        if self.scale_aligned:
            s2 = self.s2_scale(s2)
            s1 = self.s1_scale(s1)
            topo = self.srtm_scale(topo)
        s2_s1_srtm = torch.cat((s2, s1, topo), 1)
        return self.out(s2_s1_srtm)


class UpConvBlock(nn.Module):
    def __init__(self,
                 conv_block,
                 in_channels,
                 skip_channels,
                 out_channels,
                 num_convs=2,
                 stride=1,
                 dilation=1,
                 with_cp=False,
                 conv_cfg=None,
                 norm_cfg=dict(type='BN'),
                 act_cfg=dict(type='ReLU'),
                 upsample_cfg=dict(type='InterpConv'),
                 dcn=None,
                 plugins=None,
                 add_feature_map=False):
        super(UpConvBlock, self).__init__()
        assert dcn is None, 'Not implemented yet.'
        assert plugins is None, 'Not implemented yet.'

        self.add_feature_map = add_feature_map
        if not add_feature_map:
            self.conv_block = conv_block(
                in_channels=2 * skip_channels,
                out_channels=out_channels,
                num_convs=num_convs,
                stride=stride,
                dilation=dilation,
                with_cp=with_cp,
                conv_cfg=conv_cfg,
                norm_cfg=norm_cfg,
                act_cfg=act_cfg,
                dcn=None,
                plugins=None)
        if upsample_cfg is not None:
            self.upsample = build_upsample_layer(
                cfg=upsample_cfg,
                in_channels=in_channels,
                out_channels=skip_channels,
                with_cp=with_cp,
                norm_cfg=norm_cfg,
                act_cfg=act_cfg)
        else:
            self.upsample = ConvModule(
                in_channels,
                skip_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                conv_cfg=conv_cfg,
                norm_cfg=norm_cfg,
                act_cfg=act_cfg)

    def forward(self, skip, x):
        x = self.upsample(x)
        if self.add_feature_map:
            out = skip + x
        else:
            out = torch.cat([skip, x], dim=1)
            out = self.conv_block(out)
        return out


class BaseDecodeHead(BaseModule, metaclass=ABCMeta):
    def __init__(self,
                 in_channels,
                 channels,
                 *,
                 num_classes,
                 out_channels=None,
                 threshold=None,
                 dropout_ratio=0.1,
                 conv_cfg=None,
                 norm_cfg=None,
                 act_cfg=dict(type='ReLU'),
                 in_index=-1,
                 input_transform=None,
                 loss_decode=dict(
                     type='CrossEntropyLoss',
                     use_sigmoid=False,
                     loss_weight=1.0),
                 ignore_index=255,
                 sampler=None,
                 align_corners=False,
                 init_cfg=dict(
                     type='Normal', std=0.01, override=dict(name='conv_seg'))):
        super(BaseDecodeHead, self).__init__(init_cfg)
        self._init_inputs(in_channels, in_index, input_transform)
        self.channels = channels
        self.dropout_ratio = dropout_ratio
        self.conv_cfg = conv_cfg
        self.norm_cfg = norm_cfg
        self.act_cfg = act_cfg
        self.in_index = in_index

        self.ignore_index = ignore_index
        self.align_corners = align_corners

        if out_channels is None:
            if num_classes == 2:
                warnings.warn('For binary segmentation, we suggest using'
                              '`out_channels = 1` to define the output'
                              'channels of segmentor, and use `threshold`'
                              'to convert seg_logist into a prediction'
                              'applying a threshold')
            out_channels = num_classes

        if out_channels != num_classes and out_channels != 1:
            raise ValueError(
                'out_channels should be equal to num_classes,'
                'except binary segmentation set out_channels == 1 and'
                f'num_classes == 2, but got out_channels={out_channels}'
                f'and num_classes={num_classes}')

        if out_channels == 1 and threshold is None:
            threshold = 0.3
            warnings.warn('threshold is not defined for binary, and defaults'
                          'to 0.3')
        self.num_classes = num_classes
        self.out_channels = out_channels
        self.threshold = threshold

        if sampler is not None:
            self.sampler = build_pixel_sampler(sampler, context=self)
        else:
            self.sampler = None

        self.conv_seg = nn.Conv2d(channels, self.out_channels, kernel_size=1)
        if dropout_ratio > 0:
            self.dropout = nn.Dropout2d(dropout_ratio)
        else:
            self.dropout = None
        self.fp16_enabled = False

    def extra_repr(self):
        """Extra repr."""
        s = f'input_transform={self.input_transform}, ' \
            f'ignore_index={self.ignore_index}, ' \
            f'align_corners={self.align_corners}'
        return s

    def _init_inputs(self, in_channels, in_index, input_transform):
        if input_transform is not None:
            assert input_transform in ['resize_concat', 'multiple_select']
        self.input_transform = input_transform
        self.in_index = in_index
        if input_transform is not None:
            assert isinstance(in_channels, (list, tuple))
            assert isinstance(in_index, (list, tuple))
            assert len(in_channels) == len(in_index)
            if input_transform == 'resize_concat':
                self.in_channels = sum(in_channels)
            else:
                self.in_channels = in_channels
        else:
            assert isinstance(in_channels, int)
            assert isinstance(in_index, int)
            self.in_channels = in_channels

    def _transform_inputs(self, inputs):
        if self.input_transform == 'resize_concat':
            inputs = [inputs[i] for i in self.in_index]
            upsampled_inputs = [
                resize(
                    input=x,
                    size=inputs[0].shape[2:],
                    mode='bilinear',
                    align_corners=self.align_corners) for x in inputs
            ]
            inputs = torch.cat(upsampled_inputs, dim=1)
        elif self.input_transform == 'multiple_select':
            inputs = [inputs[i] for i in self.in_index]
        else:
            inputs = inputs[self.in_index]

        return inputs

    @abstractmethod
    def forward(self, inputs):
        pass

    def forward_test(self, inputs, img_metas, test_cfg):
        return self.forward(inputs)

    def cls_seg(self, feat):
        if self.dropout is not None:
            feat = self.dropout(feat)
        output = self.conv_seg(feat)
        return output


class PSPHead(BaseDecodeHead):
    def __init__(self, pool_scales=(1, 2, 3, 6), **kwargs):
        super(PSPHead, self).__init__(**kwargs)
        assert isinstance(pool_scales, (list, tuple))
        self.pool_scales = pool_scales
        self.psp_modules = PPM(
            self.pool_scales,
            self.in_channels,
            self.channels,
            conv_cfg=self.conv_cfg,
            norm_cfg=self.norm_cfg,
            act_cfg=self.act_cfg,
            align_corners=self.align_corners)
        self.bottleneck = ConvModule(
            self.in_channels + len(pool_scales) * self.channels,
            self.channels,
            3,
            padding=1,
            conv_cfg=self.conv_cfg,
            norm_cfg=self.norm_cfg,
            act_cfg=self.act_cfg)

    def _forward_feature(self, inputs):
        x = self._transform_inputs(inputs)
        psp_outs = [x]
        psp_outs.extend(self.psp_modules(x))
        psp_outs = torch.cat(psp_outs, dim=1)
        feats = self.bottleneck(psp_outs)
        return feats

    def forward(self, inputs):
        output = self._forward_feature(inputs)
        output = self.cls_seg(output)
        return output


class PPM(nn.ModuleList):
    def __init__(self, pool_scales, in_channels, channels, conv_cfg, norm_cfg,
                 act_cfg, align_corners, **kwargs):
        super(PPM, self).__init__()
        self.pool_scales = pool_scales
        self.align_corners = align_corners
        self.in_channels = in_channels
        self.channels = channels
        self.conv_cfg = conv_cfg
        self.norm_cfg = norm_cfg
        self.act_cfg = act_cfg
        for pool_scale in pool_scales:
            self.append(
                nn.Sequential(
                    nn.AdaptiveAvgPool2d(pool_scale),
                    ConvModule(
                        self.in_channels,
                        self.channels,
                        1,
                        conv_cfg=self.conv_cfg,
                        norm_cfg=self.norm_cfg,
                        act_cfg=self.act_cfg,
                        **kwargs)))

    def forward(self, x):
        ppm_outs = []
        for ppm in self:
            ppm_out = ppm(x)
            upsampled_ppm_out = resize(
                ppm_out,
                size=x.size()[2:],
                mode='bilinear',
                align_corners=self.align_corners)
            ppm_outs.append(upsampled_ppm_out)
        return ppm_outs


# Main

Main Section:
This section carries out the global urban mapping task. It involves the following steps:
- Loading the necessary multimodal dataset and the pretrained model.
- Using the loaded model to predict urban areas within the provided multimodal data.

## Load data and model

In [ ]:
# Prepare the multimodal dataset for prediction.
init_default_scope('mmseg')
inference_dataset = GUM(INFERENCE_DATASET_DIR, with_product=with_product, transform=data_transforms, inference_mode=True)
inference_loader = DataLoader(inference_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False, timeout=120)

# Initialize the GUM 2.0 for inference.
model = UNetResMultiEnc()

checkpoint = torch.load(CKPT_PATH, map_location=lambda storage, loc: storage)
checkpoint_state_dict = checkpoint['state_dict']
model.load_state_dict(checkpoint_state_dict, strict=False)
model.to(DEVICE)

## Prediction

In [ ]:
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    for inputs, img_metas in tqdm(inference_loader, desc='Inference Progress', unit='batch'):
        inputs = inputs.to(DEVICE)

        s2, s1, topo = inputs[:, :4, :, :], inputs[:, 4:8, :, :], inputs[:, 8:, :, :]
        outputs = model.test_step(s2, s1, topo)

        file_name = img_metas['filename'][0] # Assuming img_meta is a list of lists of dicts
        input_path = os.path.join(INFERENCE_DATASET_DIR, file_name)
        output_path = os.path.join(PREDICTED_DATASET_DIR, file_name)
        with rasterio.open(input_path) as src:
            profile = src.profile

        output = outputs.data.cpu().numpy()

        profile.update(
            dtype=np.uint8,
            nodata=255,
            count=1  # Update the number of bands to the output's number of bands
        )

        # Save the prediction
        if output.ndim == 2:
            output = output[np.newaxis, :, :]

        # Write the output with the same profile as the input image
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(output)